# Домашнее задание. Цена подержанных BMW: подготовка данных и линейная регрессия тремя способами

Ты работаешь в сервисе объявлений о продаже машин. Продавцы часто не знают, какую цену поставить, и сервис хочет подсказывать им рыночную цену. Твоя задача — построить модель линейной регрессии, которая предсказывает цену подержанной BMW по её характеристикам.

Задание состоит из пяти частей:

1. **Подготовка данных** — разведка, очистка, выбросы, новые признаки, one-hot encoding.
2. **Разбиение** на обучающую и тестовую выборки и стандартизация.
3. **Поиск весов** линейной регрессии тремя способами: точная формула, градиентный спуск, мини-батчевый градиентный спуск.
4. **Сравнение** методов по времени и метрикам.
5. **Выводы.**

**Правила**

- Код пиши сам. В каждой ячейке расписано по шагам, что нужно сделать, — пиши код под комментариями. Если не хватает места, добавляй ячейки.
- Для моделей используй только `numpy`. Готовые модели из `scikit-learn` использовать **нельзя** — веса нужно найти самому. Из `scikit-learn` можно взять только `train_test_split`, если хочешь.
- Каждое решение в подготовке данных объясняй в ячейке **«Твой ответ»**: что сделал и почему. Важно не только сделать, но и обосновать.
- Перед сдачей выполни `Kernel → Restart & Run All` и проверь, что ноутбук проходит сверху вниз без ошибок.

## Данные

Датасет: [100,000 UK Used Car Data set](https://www.kaggle.com/datasets/adityadesai13/used-car-dataset-ford-and-mercedes?select=bmw.csv) на Kaggle, файл **bmw.csv** — около 10 тысяч объявлений о продаже подержанных BMW в Великобритании. Данные собраны в 2020 году.

Скачай файл `bmw.csv` и положи в одну папку с ноутбуком.

| столбец | описание |
|---|---|
| `model` | модель BMW: 3 Series, 1 Series, X5 и т.д. |
| `year` | год регистрации машины |
| `price` | цена, £ — **целевая переменная** |
| `transmission` | коробка передач: Automatic, Semi-Auto, Manual |
| `mileage` | пробег, мили |
| `fuelType` | тип топлива: Diesel, Petrol, Hybrid и др. |
| `tax` | годовой дорожный налог, £ |
| `mpg` | расход: сколько миль машина проезжает на одном галлоне топлива |
| `engineSize` | объём двигателя, л |

# Часть 1. Подготовка данных

In [ ]:
# Импортируй numpy, pandas, matplotlib.pyplot и модуль time
# Прочитай bmw.csv в переменную df
# Выведи размер таблицы и первые 10 строк

### 1.1. Разведка данных

Прежде чем что-то чистить, нужно понять, что лежит в данных.

In [ ]:
# Выведи df.info(): какие типы у столбцов, есть ли пропуски
# Выведи df.describe() для числовых столбцов
# Для каждого категориального столбца (model, transmission, fuelType) выведи value_counts()
# Посмотри внимательно на значения категорий: нет ли лишних пробелов в начале или в конце строк?
#     (подсказка: выведи df['model'].unique() и посмотри на кавычки)

In [ ]:
# Построй гистограммы для price, mileage, mpg, engineSize, tax, year
#     (подсказка: df[[...]].hist(bins=50, figsize=(14, 8)))
# Построй scatter: year по оси x, price по оси y; mileage по оси x, price по оси y

**Вопросы:** какие признаки, судя по графикам, сильнее всего связаны с ценой? Какие значения выглядят подозрительно (например, нулевой объём двигателя или очень большой mpg)? Какие категории встречаются очень редко?

**Твой ответ:**

_..._

### 1.2. Пробелы в строках, пропуски и дубликаты

In [ ]:
# Убери лишние пробелы в начале и конце строк во всех категориальных столбцах (метод .str.strip())
# Посчитай число пропусков в каждом столбце. Если пропуски есть — удали строки с ними (dropna)
# Посчитай число полных дубликатов строк (df.duplicated().sum())
# Удали дубликаты (drop_duplicates) и выведи, сколько строк осталось

**Вопрос:** почему дубликаты нужно удалять до разбиения на train и test? Что случится, если одна и та же строка попадёт в обе выборки?

**Твой ответ:**

_..._

### 1.3. Выбросы и некорректные значения

Выброс — объект, который сильно отличается от остальных. Это может быть ошибка в данных (объём двигателя 0 литров у бензиновой машины) или редкий, но реальный объект (электромобиль с огромным mpg, машина за £120 000). Линейная регрессия чувствительна к выбросам: один экстремальный объект может сильно повернуть веса.

Для каждого числового признака реши, что считать выбросом и что с ним делать. Варианты: удалить строки, ограничить значения (обрезать по квантилю) или оставить, если это реальные данные. Каждое решение обоснуй.

Полезные инструменты:

- квантили: `df['price'].quantile([0.01, 0.99])`;
- правило IQR: выбросом считается значение ниже $Q_1 - 1.5 \cdot IQR$ или выше $Q_3 + 1.5 \cdot IQR$, где $IQR = Q_3 - Q_1$;
- `plt.boxplot(...)` — ящик с усами, точки за усами — кандидаты в выбросы.

In [ ]:
# Построй boxplot для price, mileage, mpg, engineSize
# Посчитай, сколько машин с engineSize == 0 и какие у них fuelType. Реши, что с ними делать
# Посмотри на машины с самым большим mpg: какой у них fuelType? Реальные ли это значения?
# Посмотри на самые дорогие машины: что это за модели и годы?
# Прими решение по выбросам и примени его
# Выведи, сколько строк было до и после очистки от выбросов

**Вопросы:** какие выбросы ты удалил и почему? Какие оставил и почему? Какую долю данных ты потерял?

**Твой ответ:**

_..._

### 1.4. Редкие категории

In [ ]:
# Посмотри ещё раз на value_counts() для model, transmission и fuelType
# Категории, которые встречаются очень редко (например, меньше 10–20 раз), объедини в одну категорию 'Other'
#     или удали эти строки — выбери вариант и обоснуй
# Выведи value_counts() после изменений

**Вопрос:** почему редкие категории — проблема для модели? Сколько объектов нужно, чтобы модель могла надёжно оценить вес такой категории?

**Твой ответ:**

_..._

### 1.5. Feature engineering и удаление лишних признаков

Feature engineering — создание новых признаков из имеющихся, чтобы модели было проще уловить закономерность. Линейная модель умеет только складывать признаки с весами, поэтому удачно придуманный признак может сильно улучшить качество.

Идеи, которые стоит проверить:

- **возраст машины** вместо года: `age = 2020 - year` (данные собраны в 2020 году). Подумай, нужен ли после этого исходный `year`;
- **пробег в год**: `mileage / age` — интенсивно ли машину эксплуатировали (осторожно с делением на ноль у машин 2020 года);
- можно придумать свои признаки.

Подумай также, какие признаки лишние: дублируют другие, не несут информации или были бы недоступны в момент, когда продавец создаёт объявление.

In [ ]:
# Создай новые признаки, которые считаешь полезными
# Посчитай корреляцию числовых признаков с price (df.corr(numeric_only=True)['price'])
# Посчитай корреляцию признаков между собой: нет ли пар, которые почти дублируют друг друга (|корреляция| > 0.9)?
# Удали лишние признаки

**Вопросы:** какие признаки ты создал и почему они должны помочь? Какие удалил и почему? Почему два почти одинаковых признака — проблема для точной формулы весов?

**Твой ответ:**

_..._

### 1.6. One-hot encoding

Категориальные признаки (`model`, `transmission`, `fuelType`) нужно превратить в числа. Используй one-hot encoding с удалением одной категории: если у признака $N$ вариантов, получится $N - 1$ столбец из 0 и 1.

In [ ]:
# Сделай one-hot encoding категориальных признаков: pd.get_dummies(..., drop_first=True, dtype=int)
# Выведи, сколько столбцов получилось и какие
# Проверь, что в таблице не осталось нечисловых столбцов

**Вопрос:** почему нужно удалять одну категорию (drop_first=True), если в модели есть свободный член $w_0$? Ты проверишь это на практике в задании 3.2.

**Твой ответ:**

_..._

# Часть 2. Разбиение и стандартизация

### 2.1. Разбиение на train и test

Раздели данные: 80% — обучающая выборка, 20% — тестовая. Зафиксируй случайность (`random_state=42` или `np.random.default_rng(42)`), чтобы результаты воспроизводились.

**С этого момента тестовую выборку используем только для финальной оценки качества.** Никакие параметры (средние, стандартные отклонения, пороги) по ней не считаем.

In [ ]:
# Отдели целевую переменную price от признаков
# Раздели данные на X_train, X_test, y_train, y_test в пропорции 80/20
# Выведи размеры всех четырёх частей

### 2.2. Стандартизация

Стандартизация приводит каждый признак к среднему 0 и стандартному отклонению 1:

$$x_{\text{new}} = \frac{x - \mu}{\sigma}$$

Для точной формулы она не обязательна, а для градиентного спуска — очень нужна: без неё признаки в разных масштабах (пробег в десятках тысяч, объём двигателя — единицы) не дадут спуску нормально сойтись.

In [ ]:
# Реши, какие столбцы стандартизировать. Нужно ли стандартизировать столбцы one-hot из 0 и 1? Обоснуй
# Посчитай среднее и стандартное отклонение по обучающей выборке
# Примени стандартизацию к train и к test, используя параметры, посчитанные по train
# Подумай, стандартизировать ли целевую переменную y. Если да — не забудь вернуть предсказания
#     в фунты перед подсчётом метрик
# Добавь в начало X_train и X_test столбец из единиц для свободного члена
# Переведи всё в numpy-массивы и выведи размеры: сколько объектов и сколько признаков вместе со столбцом единиц

**Вопрос:** почему среднее и стандартное отклонение считают только по обучающей выборке? Как называется ошибка, если посчитать их по всем данным?

**Твой ответ:**

_..._

# Часть 3. Линейная регрессия: поиск весов

### 3.0. Функция метрик

Для всех методов будем считать одни и те же метрики на **тестовой** выборке.

In [ ]:
# Напиши функцию metrics(y_true, y_pred), которая возвращает словарь с метриками:
#     MAE  — средняя абсолютная ошибка, £
#     RMSE — корень из средней квадратичной ошибки, £
#     MAPE — средняя абсолютная ошибка в процентах от реальной цены
#     R²   — коэффициент детерминации
# Посчитай метрики бейзлайна: предсказываем для всех машин из теста среднюю цену из train
#     Любая модель должна быть заметно лучше бейзлайна

### 3.1. Способ 1. Точная формула

Веса линейной регрессии, минимизирующие среднеквадратичную ошибку, считаются по формуле нормального уравнения:

$$w = (X^T X)^{-1} X^T y$$

In [ ]:
# Засеки время: t0 = time.perf_counter()
# Посчитай веса по формуле (np.linalg.inv, транспонирование .T, матричное умножение @)
# Засеки время окончания и выведи, сколько секунд занял расчёт
# Выведи размер матрицы X^T X
# Посчитай предсказания на тесте и метрики
# Выведи веса вместе с названиями признаков. Какие признаки сильнее всего влияют на цену?

### 3.2. Когда точная формула ломается: ловушка дамми-переменных

Проверь на практике, что будет, если сделать one-hot encoding **без** удаления одной категории.

In [ ]:
# Сделай копию данных и one-hot encoding с drop_first=False (все N столбцов на каждую категорию)
# Подготовь X точно так же, как раньше: та же стандартизация числовых признаков, столбец единиц
# Попробуй посчитать веса по точной формуле. Оберни расчёт в try / except np.linalg.LinAlgError
#     и выведи текст ошибки, если она возникнет
# Посчитай ранг матрицы X (np.linalg.matrix_rank) и сравни с числом столбцов
# Если ошибки не было — посмотри на веса и метрики: адекватные ли они?

**Вопрос:** что произошло и почему? Как связаны сумма one-hot столбцов одной категории и столбец из единиц?

**Твой ответ:**

_..._

### 3.3. Когда точная формула ломается: много признаков

На наших данных объектов около 10 тысяч, а признаков несколько десятков, поэтому матрица $X^T X$ маленькая и точная формула считается за миллисекунды. Но обращение матрицы размера $d \times d$ стоит порядка $d^3$ операций, а вычисление $X^T X$ — порядка $n \cdot d^2$. Когда признаков тысячи или десятки тысяч (так бывает, например, когда признаки — слова в тексте или пиксели картинки), формула начинает работать очень долго и требовать много памяти.

Проверим, как растёт время, если к нашим признакам добавить много дополнительных случайных признаков. **Порог времени:** если очередной расчёт занял больше `TIME_LIMIT` секунд, следующий размер не запускаем и считаем, что формула «не справилась» — на реальных задачах признаков ещё на порядки больше.

Осторожно: при больших `d` матрица занимает много памяти. Не запускай размеры больше 5 000.

In [ ]:
# Задай TIME_LIMIT = 2 (секунды) и список размеров d_list = [50, 200, 500, 1000, 2000, 3000, 5000]
# Для каждого d из списка:
#     создай матрицу из d случайных признаков для обучающих объектов (np.random.default_rng(42).normal(size=(n_train, d)))
#     приклей её справа к X_train (np.hstack)
#     засеки время расчёта весов по точной формуле
#     сохрани d, полное число признаков и время
#     если время больше TIME_LIMIT — выведи сообщение «формула не справилась на d признаках» и прерви цикл (break)
# Выведи таблицу результатов и построй график: число признаков по оси x, время по оси y

**Вопросы:** при каком числе признаков время превысило порог? Во сколько раз выросло время, когда число признаков выросло в 2 раза? Совпадает ли это с оценкой $d^3$? Что выгоднее для таких задач — точная формула или градиентный спуск?

**Твой ответ:**

_..._

### 3.4. Способ 2. Градиентный спуск

Функция потерь — среднеквадратичная ошибка:

$$L(w) = \frac{1}{n}\sum_{i=1}^{n}(x_i^T w - y_i)^2$$

Её градиент по весам:

$$\nabla L(w) = \frac{2}{n} X^T (Xw - y)$$

Шаг градиентного спуска:

$$w \leftarrow w - \eta \cdot \nabla L(w)$$

где $\eta$ — шаг обучения (learning rate). На каждой итерации градиент считается по **всей** обучающей выборке.

In [ ]:
# Напиши функцию gradient_descent(X, y, lr, tol, max_iter), которая:
#     начинает с нулевых весов
#     на каждой итерации считает градиент по всей выборке и делает шаг
#     после каждого шага считает MSE на обучающей выборке и сохраняет её в список history
#     останавливается, когда относительное уменьшение MSE стало меньше tol, или после max_iter итераций
#     возвращает веса, число итераций и history
# Подбери шаг обучения: попробуй несколько значений (например, 0.001, 0.01, 0.1, 0.5) и посмотри,
#     при каком спуск сходится быстрее всего и не расходится

In [ ]:
# Запусти градиентный спуск с выбранным шагом и засеки время
# Выведи время, число итераций и метрики на тесте
# Сравни веса с весами точной формулы: посчитай максимальную разницу np.abs(w_gd - w_exact).max()
# Построй график history: номер итерации по оси x, MSE по оси y (логарифмическая шкала plt.yscale('log'))

**Вопросы:** что происходит при слишком большом и слишком маленьком шаге обучения? Совпали ли веса с точной формулой и почему они не совпадают идеально?

**Твой ответ:**

_..._

### 3.5. Способ 3. Мини-батчевый градиентный спуск

В мини-батчевом градиентном спуске (mini-batch gradient descent) градиент на каждом шаге считается не по всей выборке, а по небольшой случайной порции объектов — **батчу**.

- **итерация** — один шаг обновления весов по одному батчу;
- **эпоха** — полный проход по всей обучающей выборке. При 8 000 объектах и батче 64 одна эпоха — это 125 итераций.

Если батч состоит из одного объекта, метод называют стохастическим градиентным спуском (SGD).

Из-за случайности батчей веса около минимума «дрожат», поэтому шаг обучения часто постепенно уменьшают, например умножают на 0.9 после каждой эпохи.

In [ ]:
# Напиши функцию minibatch_gd(X, y, lr, batch_size, tol, max_epochs), которая:
#     начинает с нулевых весов
#     в начале каждой эпохи перемешивает индексы объектов (rng.permutation)
#     проходит по батчам и после каждого батча обновляет веса по градиенту, посчитанному только по этому батчу
#     считает общее число итераций (обновлений весов)
#     в конце каждой эпохи считает MSE на всей обучающей выборке и сохраняет её в history
#     останавливается, когда относительное уменьшение MSE за эпоху стало меньше tol, или после max_epochs эпох
#     (по желанию) уменьшает шаг обучения после каждой эпохи
#     возвращает веса, число итераций, число эпох и history

In [ ]:
# Запусти мини-батчевый спуск с batch_size=64 и засеки время
# Выведи время, число эпох, число итераций и метрики на тесте
# Сравни веса с весами точной формулы
# Попробуй batch_size 16, 64, 256, 1024: как меняются время, число эпох и итераций, качество?

In [ ]:
# Построй на одном графике history полного градиентного спуска и мини-батчевого по эпохам
#     (для полного спуска одна итерация = одна эпоха)

# Часть 4. Сравнение методов

In [ ]:
# Собери итоговую таблицу (pd.DataFrame), где строки — методы:
#     бейзлайн, точная формула, градиентный спуск, мини-батчевый градиентный спуск
# а столбцы:
#     время поиска весов, число итераций, число эпох,
#     сколько объектов суммарно обработано при расчёте градиентов (итерации × размер батча),
#     MAE, RMSE, MAPE, R² на тесте
# Выведи таблицу

In [ ]:
# Построй столбчатую диаграмму времени работы трёх методов
# Построй столбчатую диаграмму весов трёх методов для нескольких самых важных признаков

# Часть 5. Выводы

Ответь на вопросы, опираясь на свои результаты. Пиши конкретно, с числами из своих таблиц.

1. Какие шаги подготовки данных сильнее всего повлияли на качество модели? Как ты это проверил?
2. Какой метод нашёл веса быстрее всего на этом датасете? Почему?
3. Отличается ли качество моделей, обученных тремя способами? Почему?
4. Почему мини-батчевый спуск делает больше итераций, чем полный, но может обработать меньше объектов? Что выгоднее при миллионах объектов?
5. В каких ситуациях точная формула не подходит? Приведи две причины из своих экспериментов в 3.2 и 3.3.
6. Насколько хорошо модель предсказывает цену? Посмотри на MAE и MAPE: можно ли с такой ошибкой подсказывать цену продавцам?
7. Что ещё можно сделать, чтобы улучшить модель? Например, подумай, стоит ли предсказывать не саму цену, а её логарифм.

**Твой ответ:**

_..._